# 03 · Deep Agents 接入

使用相同模型、工具、资料和任务，对照普通 Agent。

本地 Deep Agents 0.7.38 需要显式加入 `TodoListMiddleware`；不能照搬早期版本“默认带规划”的说法。
本实验不配置子 Agent、Skills 或长期记忆，也不调用默认文件工具。

In [ ]:
from planning_lab import (
    initialize, live_model, TOOLS, SYSTEM_PROMPT, REQUEST, FOLLOWUP,
    read_requirements, read_constraints, observe_update, final_answer,
)
initialize()
# 保持 False 可从头执行资料与 schema 检查，不调用模型。
# 确认密钥后手动改成 True，再从头运行。本章不会使用模拟模型。
RUN_LIVE = False
print(read_requirements.invoke({}))
print(read_constraints.invoke({}))
for tool in TOOLS:
    print(tool.name, tool.args_schema.model_json_schema())

## 限制本实验向模型开放的工具

Deep Agents 默认还有文件与委派工具；`subagents=[]` 不等于禁用默认通用子 Agent。
下面通过模型调用中间件只开放本篇的三个工具，不去修改全局 profile。

这段是教学实验的工具选择配置，不是 Todo 规划的必需代码。

In [ ]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware, wrap_model_call

ALLOWED_TOOLS = {'write_todos', 'read_requirements', 'read_constraints'}

@wrap_model_call
def only_chapter_tools(request, handler):
    selected = [tool for tool in request.tools if tool.name in ALLOWED_TOOLS]
    assert {tool.name for tool in selected} == ALLOWED_TOOLS
    return handler(request.override(tools=selected))

## 创建并运行 Deep Agent（真实调用）

In [ ]:
if RUN_LIVE:
    model = live_model()
    agent = create_deep_agent(
        model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
        middleware=[TodoListMiddleware(), only_chapter_tools],
    )
    trace = {'messages': [], 'snapshots': [], 'todos': []}
    for update in agent.stream(
        {'messages': [{'role': 'user', 'content': REQUEST}]},
        {'recursion_limit': 60},
        stream_mode='updates',
    ):
        observe_update(update, trace)
    print('最终待办：', trace['todos'])
    print('最终方案：', final_answer(trace))
else:
    print('已跳过真实模型实验。')

## 与普通 Agent 比较

- 仍然是模型调用 `write_todos`、业务工具，接着更新计划并输出成果。
- 比较两个 Notebook 的真实轨迹即可，不要求任务名称、条数或调用顺序完全一致。
- 默认其他工具在本实验中不向模型开放；没有通过委派来完成资料分析。
- Deep Agents 的额外能力没有将 Todo 变成自动调度的执行图；要强制依赖、顺序和验收，需要另外设计工作流。
- 如果后续版本改变工具或中间件行为，工具集合断言会明确失败；先核对版本，不要删掉断言后声称实验不变。